In [1]:
import pandas as pd
import numpy as np
import math
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
S3_BUCKET = "ads508-housing-data-faye"
DATA_PATH = "s3://ads508-housing-data-faye/prepared/full_prepared_dataset.csv"

df = pd.read_csv(DATA_PATH)

In [4]:
 df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
print(df.shape)
df.head()

(82786, 17)


,RegionID,SizeRank,RegionName,RegionType,StateName,Date,ZHVI,Inventory,MarketTemp,MedianSalePrice,Year,Month,ZHVI_Lag1,Inventory_Lag1,MarketTemp_Lag1,ZHVI_PctChange,Inventory_PctChange
0,394297,677,"Aberdeen, SD",msa,SD,2018-04-30,151243.332163,205.0,44.0,NaN,2018,4,150947.408155,189.0,41.0,0.001960,0.084656
1,394297,677,"Aberdeen, SD",msa,SD,2018-05-31,151561.875115,231.0,50.0,NaN,2018,5,151243.332163,205.0,44.0,0.002106,0.126829
2,394297,677,"Aberdeen, SD",msa,SD,2018-06-30,151737.806399,263.0,52.0,NaN,2018,6,151561.875115,231.0,50.0,0.001161,0.138528
3,394297,677,"Aberdeen, SD",msa,SD,2018-07-31,152341.825731,294.0,51.0,NaN,2018,7,151737.806399,263.0,52.0,0.003981,0.117871
4,394297,677,"Aberdeen, SD",msa,SD,2018-08-31,152652.372110,309.0,48.0,NaN,2018,8,152341.825731,294.0,51.0,0.002038,0.051020


In [5]:
feature_cols = [
    "StateName", "Year", "Month", "Inventory", "MarketTemp", "MedianSalePrice",
    "ZHVI_Lag1", "Inventory_Lag1", "MarketTemp_Lag1",
    "ZHVI_PctChange", "Inventory_PctChange"
]

target_col = "ZHVI"

df = df.dropna(subset=[target_col]).copy()
X = df[feature_cols]
y = df[target_col]

print(X.shape, y.shape)

(82786, 11) (82786,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (66228, 11)
Testing shape: (16558, 11)


In [12]:
numeric_features = [c for c in feature_cols if c != "StateName"]
categorical_features = ["StateName"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=1
    ))
])

In [8]:
model.fit(X_train, y_train)
preds = model.predict(X_test)
print("Model training complete.")

Model training complete.


In [9]:
mae = mean_absolute_error(y_test, preds)
rmse = math.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

MAE: 229.31
RMSE: 739.45
R2 Score: 1.0000


In [10]:
results = pd.DataFrame({
    "Actual_ZHVI": y_test.values[:10],
    "Predicted_ZHVI": preds[:10]
})

results

,Actual_ZHVI,Predicted_ZHVI
0,231431.088119,231334.602958
1,300293.813063,300005.571069
2,258124.420473,258033.658200
3,176112.297238,176404.593646
4,130338.808877,130171.455317
5,549478.670301,548482.765405
6,93718.292173,93177.835805
7,133485.298327,133418.197668
8,172459.508999,172534.548441
9,183072.828609,183076.875900


In [13]:
joblib.dump(model, "module5_rf_model.joblib")
print("Model saved as module5_rf_model.joblib")

Model saved as module5_rf_model.joblib
